# CardioSense - Baseline Arrhythmia Model (Random Forest, no TensorFlow)

This notebook builds a multi-class arrhythmia beat classifier without TensorFlow.

Pipeline:
1. Download three PhysioNet databases (mitdb, svdb, incartdb) to spread beat classes more evenly
2. Extract a fixed-length window around every labeled beat, resampled to 250Hz to match the ESP32 firmware sample rate
3. Turn each window into a small set of numeric features (not raw waveform) - this is what makes a Random Forest a good fit instead of a CNN
4. Train a Random Forest classifier on the AAMI beat categories (N, S, V, F, Q)
5. Export the trained model directly to a C header file using `emlearn` - no TFLite, no runtime library, just plain C that compiles straight into the firmware

Run cells top to bottom. The database download step only needs to run once - after that, `wfdb` reads from the local `data/` folder.

In [17]:
import os
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

## Step 1: Download the databases

Three sources, each in its own subfolder so record names never collide:
- **mitdb** (360Hz) - the main dataset, but heavily skewed toward Normal beats
- **svdb** (128Hz) - adds far more Supraventricular (S) examples
- **incartdb** (257Hz) - adds more Ventricular (V) examples from a different patient population

This step downloads every record in each database. It only needs to run once - re-running it just confirms the files are already there.

## Step 2: Map beat symbols to AAMI classes

PhysioNet labels individual beats with detailed symbols (e.g. 'N', 'L', 'A', 'V'...).
The AAMI standard groups these into 5 clinically meaningful classes:

- **N** - Normal
- **S** - Supraventricular ectopic beat
- **V** - Ventricular ectopic beat
- **F** - Fusion beat
- **Q** - Unknown / paced beat

Any symbol not in this map (noise markers, non-beat annotations) is skipped.

In [18]:
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    'P': 'Q', '/': 'Q', 'f': 'Q', 'u': 'Q',
}

## Step 3: Beat window extraction + feature engineering

Each beat window is 0.4s before the R-peak and 0.6s after, resampled to 250 samples at 250Hz -
matching the firmware's ADC sample rate exactly, so real device data will look like this too.

Instead of feeding the raw 250-sample window into a model (which is what a CNN would need),
we reduce it to a handful of descriptive numbers. This is the key change from the TensorFlow plan:
a Random Forest works on these features directly, no waveform-shape learning required.

In [19]:
TARGET_FS = 250
WINDOW_BEFORE_S = 0.4
WINDOW_AFTER_S = 0.6
WINDOW_LEN = int((WINDOW_BEFORE_S + WINDOW_AFTER_S) * TARGET_FS)  # 250 samples


def extract_beat_window(sig, r_index, fs):
    before = int(WINDOW_BEFORE_S * fs)
    after = int(WINDOW_AFTER_S * fs)
    start = r_index - before
    end = r_index + after
    if start < 0 or end > len(sig):
        return None
    window = sig[start:end]
    return resample(window, WINDOW_LEN)


def extract_features(window, rr_prev, rr_next):
    w = pd.Series(window)
    return {
        'mean': w.mean(),
        'std': w.std(),
        'min': w.min(),
        'max': w.max(),
        'range': w.max() - w.min(),
        'skew': w.skew(),
        'kurtosis': w.kurt(),
        'energy': float(np.sum(window ** 2)),
        'peak_pos': int(np.argmax(window)),
        'rr_prev': rr_prev,
        'rr_next': rr_next,
        'rr_ratio': rr_prev / rr_next if rr_next != 0 else 0.0,
    }

In [20]:
DATABASES = {
    "mitdb": "data/mitdb",
    "svdb": "data/svdb",
    "incartdb": "data/incartdb",
}

## Step 4: Run extraction across every record in all three databases

This reads each record's real sample rate (they differ per database) and always reads channel 0,
since that is the single ECG lead the AD8232 setup actually has.

In [21]:
def process_database(db_name, db_dir):
    rows = []
    record_names = wfdb.get_record_list(db_name)
    total = len(record_names)

    for idx, rec_name in enumerate(record_names, start=1):
        rec_path = os.path.join(db_dir, rec_name)
        try:
            record = wfdb.rdrecord(rec_path)
            annotation = wfdb.rdann(rec_path, 'atr')
        except Exception as e:
            print(f"  [{idx}/{total}] skipping {rec_name}: {e}")
            continue

        sig = record.p_signal[:, 0]
        fs = record.fs
        r_indices = annotation.sample
        symbols = annotation.symbol

        for i in range(1, len(r_indices) - 1):
            symbol = symbols[i]
            if symbol not in AAMI_MAP:
                continue

            window = extract_beat_window(sig, r_indices[i], fs)
            if window is None:
                continue

            rr_prev = (r_indices[i] - r_indices[i - 1]) / fs
            rr_next = (r_indices[i + 1] - r_indices[i]) / fs

            feats = extract_features(window, rr_prev, rr_next)
            feats['label'] = AAMI_MAP[symbol]
            feats['source_db'] = db_name
            rows.append(feats)

        print(f"  [{idx}/{total}] {rec_name}: {len(rows)} beats so far")

    return pd.DataFrame(rows)

In [22]:
all_dfs = []
for db_name, db_dir in DATABASES.items():
    print(f"Processing {db_name} ...")
    df = process_database(db_name, db_dir)
    print(f"  {len(df)} beats extracted\n")
    all_dfs.append(df)

data = pd.concat(all_dfs, ignore_index=True)
print("Total beats:", len(data))
print(data['label'].value_counts())

Processing mitdb ...
  [1/48] 100: 2271 beats so far
  [2/48] 101: 4132 beats so far
  [3/48] 102: 6317 beats so far
  [4/48] 103: 8400 beats so far
  [5/48] 104: 10609 beats so far
  [6/48] 105: 13175 beats so far
  [7/48] 106: 15201 beats so far
  [8/48] 107: 17336 beats so far
  [9/48] 108: 19097 beats so far
  [10/48] 109: 21627 beats so far
  [11/48] 111: 23750 beats so far
  [12/48] 112: 26287 beats so far
  [13/48] 113: 28081 beats so far
  [14/48] 114: 29959 beats so far
  [15/48] 115: 31911 beats so far
  [16/48] 116: 34322 beats so far
  [17/48] 117: 35856 beats so far
  [18/48] 118: 38132 beats so far
  [19/48] 119: 40118 beats so far
  [20/48] 121: 41980 beats so far
  [21/48] 122: 44454 beats so far
  [22/48] 123: 45970 beats so far
  [23/48] 124: 47588 beats so far
  [24/48] 200: 50188 beats so far
  [25/48] 201: 52150 beats so far
  [26/48] 202: 54285 beats so far
  [27/48] 203: 57259 beats so far
  [28/48] 205: 59914 beats so far
  [29/48] 207: 61772 beats so far
  [30/

## Step 5: Train the Random Forest

`class_weight='balanced'` tells the model to pay proportionally more attention to the rare classes
(S, F, Q) instead of just optimizing for the dominant N class - directly addressing the imbalance
problem from earlier.

In [23]:
clf = RandomForestClassifier(
    n_estimators=20,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",20
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_featur

## Step 6: Evaluate

Look at precision/recall **per class**, not just overall accuracy - overall accuracy can look great
while still missing the rare, clinically important classes.

In [24]:
cmodel = emlearn.convert(clf, method='loadable')
cmodel.save(file='../models/source/arrhythmia_model.h', name='arrhythmia_model')

'\n\n\n    // !!! This file is generated using emlearn !!!\n\n    #include <eml_trees.h>\n    \n\nstatic const EmlTreesNode arrhythmia_model_nodes[4744] = {\n  { 9, 0, 1, 126 },\n  { 9, 0, 1, 62 },\n  { 2, 0, 1, 32 },\n  { 6, 6, 1, 16 },\n  { 11, 0, 1, 8 },\n  { 0, 0, 1, 4 },\n  { 10, 0, 1, 2 },\n  { 6, 2, -1, -2 },\n  { 6, 4, -1, -1 },\n  { 5, 0, 1, 2 },\n  { 7, 4, -3, -1 },\n  { 9, 0, -3, -1 },\n  { 10, 0, 1, 4 },\n  { 3, 1, 1, 2 },\n  { 0, 0, -1, -1 },\n  { 7, 382, -3, -1 },\n  { 5, -1, 1, 2 },\n  { 8, 151, -3, -1 },\n  { 5, 1, -2, -4 },\n  { 1, 0, 1, 8 },\n  { 0, 0, 1, 4 },\n  { 11, 0, 1, 2 },\n  { 5, 0, -1, -3 },\n  { 0, -1, -4, -2 },\n  { 7, 5, 1, 2 },\n  { 2, 0, -3, -3 },\n  { 11, 0, -3, -3 },\n  { 5, 2, 1, 4 },\n  { 4, 3, 1, 2 },\n  { 11, 0, -1, -3 },\n  { 4, 3, -3, -3 },\n  { 0, 0, 1, 2 },\n  { 6, 15, -3, -2 },\n  { 10, 1, -3, -1 },\n  { 6, 8, 1, 16 },\n  { 2, 0, 1, 8 },\n  { 1, 0, 1, 4 },\n  { 5, 1, 1, 2 },\n  { 6, -1, -2, -1 },\n  { 3, 1, -3, -2 },\n  { 3, 2, 1, 2 },\n  { 4,

## Step 7: Export to C for the ESP32 (no TensorFlow)

`emlearn` converts the trained Random Forest directly into a C header file. No runtime library
is needed on the device - the generated code is plain, portable C99 that can be `#include`d
straight into the firmware.

Run `pip install emlearn` first if this errors on import.

In [25]:
import emlearn

os.makedirs('../models/source', exist_ok=True)

cmodel = emlearn.convert(clf, method='inline')
cmodel.save(file='../models/source/arrhythmia_model.h', name='arrhythmia_model')

# Save the label order so firmware knows how to decode the model's numeric output
with open('../models/source/label_map.txt', 'w') as f:
    for idx, label in enumerate(le.classes_):
        f.write(f"{idx}: {label}\n")

print("Exported: models/source/arrhythmia_model.h")
print("Exported: models/source/label_map.txt")

Exported: models/source/arrhythmia_model.h
Exported: models/source/label_map.txt


## Notes / next steps

- The exported `arrhythmia_model.h` takes the 12 features above (in the exact order in `feature_cols`) as input and returns the predicted class index.
- `label_map.txt` maps that index back to N/S/V/F/Q for the display module.
- The firmware still needs the R-peak detection and feature-extraction code written in C++, mirroring `extract_beat_window` / `extract_features` above, so it can produce the same 12 numbers from live sensor data before calling the model.